# Spielwiese

In [1]:
import nltk
from nltk.text import Text
from nltk.probability import FreqDist
import re
import os

In [2]:
with open("korpus\\goethe-faust2.txt", encoding="utf8") as input:
    text = input.read()
#print("Typ:",type(text))
input.close()
nltk_text = Text(nltk.word_tokenize(text))
print("Länge:",len(nltk_text))
print("NLTK:",nltk_text[42173:])
#print("TEST:",text)

Länge: 57580
NLTK: ['Luft', 'in', 'tiefste', 'Tiefen', 'bannte', ',', 'Da', ',', 'wo', 'zentralisch', 'glühend', ',', 'um', 'und', 'um', ',', 'Ein', 'ewig', 'Feuer', 'flammend', 'sich', 'durchbrannte', ',', 'Wir', 'fanden', 'uns', 'bei', 'allzugroßer', 'Hellung', 'In', 'sehr', 'gedrängter', ',', 'unbequemer', 'Stellung', '.', 'Die', 'Teufel', 'fingen', 'sämtlich', 'an', 'zu', 'husten', ',', 'Von', 'oben', 'und', 'von', 'unten', 'auszupusten', ';', 'Die', 'Hölle', 'schwoll', 'von', 'Schwefelstank', 'und', '--', 'säure', ',', 'Das', 'gab', 'ein', 'Gas', '!', 'Das', 'ging', 'ins', 'Ungeheure', ',', 'So', 'daß', 'gar', 'bald', 'der', 'Länder', 'flache', 'Kruste', ',', 'So', 'dick', 'sie', 'war', ',', 'zerkrachend', 'bersten', 'mußte', '.', 'Nun', 'haben', 'wir', "'s", 'an', 'einem', 'andern', 'Zipfel', ',', 'Was', 'ehmals', 'Grund', 'war', ',', 'ist', 'nun', 'Gipfel', '.', 'Sie', 'gründen', 'auch', 'hierauf', 'die', 'rechten', 'Lehren', ',', 'Das', 'Unterste', 'ins', 'Oberste', 'zu', 'kehr

In [72]:
print(len(set(nltk_text))/len(nltk_text))

0.2013301880610707


In [75]:
from nltk.tokenize import word_tokenize, sent_tokenize

def textanalyse(text):
    # Sätze und Wörter tokenisieren
    saetze = sent_tokenize(text)
    woerter = word_tokenize(text)

    # Wortanzahl und Satzanzahl
    anzahl_woerter = len(woerter)
    anzahl_saetze = len(saetze)

    # Zeichenanzahl (ohne Leerzeichen)
    zeichen_gesamt = sum(len(wort) for wort in woerter if wort.isalnum())

    # Durchschnittswerte
    woerter_pro_satz = anzahl_woerter / anzahl_saetze if anzahl_saetze > 0 else 0
    zeichen_pro_wort = zeichen_gesamt / anzahl_woerter if anzahl_woerter > 0 else 0

    # Ergebnisse
    return {
        "Anzahl Wörter": anzahl_woerter,
        "Anzahl Sätze": anzahl_saetze,
        "Ø Wörter pro Satz": round(woerter_pro_satz, 2),
        "Ø Zeichen pro Wort": round(zeichen_pro_wort, 2)
    }

# Beispieltext
text = "Die Ameise hatte sich verlaufen. Zum Glück fand sie den Weg zurück ins Nest."

# Analyse durchführen
ergebnisse = textanalyse(text)
for k, v in ergebnisse.items():
    print(f"{k}: {v}")


Anzahl Wörter: 16
Anzahl Sätze: 2
Ø Wörter pro Satz: 8.0
Ø Zeichen pro Wort: 3.81


In [3]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(analyzer='char', ngram_range=(3, 5))
X = vectorizer.fit_transform(["Dies ist ein Satz"])
vectorizer.get_feature_names_out()

array([' ei', ' ein', ' ein ', ' is', ' ist', ' ist ', ' sa', ' sat',
       ' satz', 'atz', 'die', 'dies', 'dies ', 'ein', 'ein ', 'ein s',
       'es ', 'es i', 'es is', 'ies', 'ies ', 'ies i', 'in ', 'in s',
       'in sa', 'ist', 'ist ', 'ist e', 'n s', 'n sa', 'n sat', 's i',
       's is', 's ist', 'sat', 'satz', 'st ', 'st e', 'st ei', 't e',
       't ei', 't ein'], dtype=object)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import jaccard_score
from sklearn.preprocessing import Binarizer
import numpy as np

def ngram_similarity(text1, text2, ngram_range=(2, 3)):
    # Vektorisierung: bi- und trigramme (wortbasiert)
    vectorizer = CountVectorizer(ngram_range=ngram_range, analyzer='word', binary=True)
    
    # Kombiniere Texte für gemeinsames Vokabular
    X = vectorizer.fit_transform([text1, text2])

    # Konvertiere Sparse-Matrix zu binärem Array (1 = N-Gramm vorhanden)
    binarizer = Binarizer()
    binarized_X = binarizer.transform(X.toarray())

    # Jaccard-Ähnlichkeit berechnen
    similarity = jaccard_score(binarized_X[0], binarized_X[1])
    return similarity

text1 = "Der Hund jagte die Katze durch den Garten."
text2 = "Die Katze rannte durch den Garten vor dem Hund davon."

similarity = ngram_similarity(text1, text2)
print(f"Jaccard-Ähnlichkeit der Bi- und Trigramme: {similarity:.2f}")


CountVectorizer(binary=True, ngram_range=(2, 3))
Jaccard-Ähnlichkeit der Bi- und Trigramme: 0.15


In [29]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import Binarizer
import numpy as np

def extract_ngram_vectors(texts, ngram_range=(2, 3)):
    """
    Wandelt eine Liste von Texten in binäre Bi-/Trigramm-Vektoren um.
    """
    vectorizer = CountVectorizer(ngram_range=ngram_range, analyzer='word', binary=True)
    X = vectorizer.fit_transform(texts)
    
    # Optional: binär machen (bei binary=True nicht notwendig, aber zur Sicherheit)
    binarizer = Binarizer()
    binary_vectors = binarizer.transform(X.toarray())
    
    return binary_vectors, vectorizer.get_feature_names_out()

texts = [
    "Der Hund jagt die Katze durch den Garten.",
    "Die Katze rennt durch den Garten vor dem Hund davon. Aber hallo und so wie geht es dir? Die ist ein sehr langer Satz der aus dem Rahmen fallen soll.",
    "Im Garten spielten Kinder mit einem Hund.",
    "Heute regnet es stark in Berlin.",
    "Und dann gibt es noch diesen Hier.",
    "Hier auch noch.",
    "Ah ja den dürfen wir nicht vergessen",
    "Zuletzt noch diesen, bei dem der Hund durch den Garten rennt und die Katze jagt."
]

vectors, ngrams = extract_ngram_vectors(texts)

# Ausgabe der N-Gramm-Feature-Vektoren
for i, vec in enumerate(vectors):
    print(f"Text {i + 1}:")
    print(vec)
    print()
print(ngrams)

Text 1:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 1 0 0 1 1 0 0 0 0
 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 1 1 0 0 0 0 0 0 0 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

Text 2:
[1 1 0 0 0 1 1 0 0 0 0 1 1 0 0 1 1 1 1 0 0 1 0 1 1 1 0 0 0 1 1 1 0 0 1 0 0
 0 1 1 1 1 0 0 1 1 0 1 1 0 0 0 0 1 0 0 0 0 1 1 1 1 0 0 1 1 0 0 0 0 1 1 0 0
 0 0 0 0 0 1 1 0 0 0 0 0 0 0 1 1 0 0 1 1 0 0 0 0 0 0 1 1 0 0 1 1 0 0 1 1 1
 1 1 1 0 0 0 0 0 0 0 0 1 1 1 1 1 1 0 0 0 0]

Text 3:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

Text 4:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0
 0 0 0 0

In [59]:
from sklearn.metrics import jaccard_score

n = len(vectors)
for i in range(n):
    for j in range(i + 1, n):
        score = jaccard_score(vectors[i], vectors[j])
        print(f"Jaccard-Ähnlichkeit Text {i+1} ↔ Text {j+1}: {score:.2f}")


Jaccard-Ähnlichkeit Text 1 ↔ Text 2: 0.06
Jaccard-Ähnlichkeit Text 1 ↔ Text 3: 0.00
Jaccard-Ähnlichkeit Text 1 ↔ Text 4: 0.00
Jaccard-Ähnlichkeit Text 1 ↔ Text 5: 0.00
Jaccard-Ähnlichkeit Text 1 ↔ Text 6: 0.00
Jaccard-Ähnlichkeit Text 1 ↔ Text 7: 0.00
Jaccard-Ähnlichkeit Text 1 ↔ Text 8: 0.14
Jaccard-Ähnlichkeit Text 2 ↔ Text 3: 0.00
Jaccard-Ähnlichkeit Text 2 ↔ Text 4: 0.00
Jaccard-Ähnlichkeit Text 2 ↔ Text 5: 0.00
Jaccard-Ähnlichkeit Text 2 ↔ Text 6: 0.00
Jaccard-Ähnlichkeit Text 2 ↔ Text 7: 0.00
Jaccard-Ähnlichkeit Text 2 ↔ Text 8: 0.05
Jaccard-Ähnlichkeit Text 3 ↔ Text 4: 0.00
Jaccard-Ähnlichkeit Text 3 ↔ Text 5: 0.00
Jaccard-Ähnlichkeit Text 3 ↔ Text 6: 0.00
Jaccard-Ähnlichkeit Text 3 ↔ Text 7: 0.00
Jaccard-Ähnlichkeit Text 3 ↔ Text 8: 0.00
Jaccard-Ähnlichkeit Text 4 ↔ Text 5: 0.00
Jaccard-Ähnlichkeit Text 4 ↔ Text 6: 0.00
Jaccard-Ähnlichkeit Text 4 ↔ Text 7: 0.00
Jaccard-Ähnlichkeit Text 4 ↔ Text 8: 0.00
Jaccard-Ähnlichkeit Text 5 ↔ Text 6: 0.00
Jaccard-Ähnlichkeit Text 5 ↔ Text 

In [53]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_distances

# 1. Schritt: Definiere einen Vectorizer, der Bi- und Trigramme extrahiert
#    – analyzer='word': Wortbasierte N-Gramme
#    – ngram_range=(2,3): Bi- und Trigramme
#    – binary=True: Präsenz (1) oder Absenz (0) jedes N-Gramms
vectorizer = CountVectorizer(analyzer='word', ngram_range=(2, 3), binary=True)

# 2. Schritt: Lies zunächst alle Texte ein, die du vergleichen möchtest
#    Beispiel: vier Test-Texte
texte = [
    "Der Hund jagte die Katze durch den Garten.",
    "Die Katze rannte durch den Garten vor dem Hund davon.",
    "Ein Auto fuhr sehr schnell die Straße entlang.",
    "Im Garten saßen zwei Kinder und spielten Verstecken.",
    "Der Hund jagte die Katze durch den Wald."
]

# 3. Schritt: Fit & Transform auf den gesamten Korpus, damit das Vokabular aller Bi-/Trigramme feststeht
#    X ist eine Sparse-Matrix der Form (Anzahl_Texte × Anzahl_Gefundener_Ngramme)
X = vectorizer.fit_transform(texte)

# 4. Schritt: Für jeden Text in 'texte' existiert nun eine Vektorentsprechung (als Zeile in X)
#    X[i] ist der Vektor (1 × n_features) für texte[i]

#    Wenn du z.B. die Feature-Namen sehen willst:
feature_names = vectorizer.get_feature_names_out()
#    Das kann nützlich sein, um nachzuvollziehen, welche Bi-/Trigramme gerade verwendet werden.
#print(feature_names)

# 5. Schritt: Beispiel: Cosine-Distanz-Matrix aller Texte untereinander berechnen
#    cosine_distances liefert eine Matrix D, in der D[i, j] = 1 – CosineSimilarity(Vektor_i, Vektor_j)
dist_matrix = cosine_distances(X)

# 6. Schritt: Ausgabe
for i, text in enumerate(texte):
    print(f"Vektor für Text {i}: (dichte Darstellung, erste 10 Dimensionen) →",
          X[i].toarray().reshape(-1)[:10], "…")  # nur zur Illustration, erste 10 Einträge

print("\nCosine-Distanz-Matrix:")
print(dist_matrix)


Vektor für Text 0: (dichte Darstellung, erste 10 Dimensionen) → [0 0 0 0 1 0 0 1 1 1] …
Vektor für Text 1: (dichte Darstellung, erste 10 Dimensionen) → [0 0 1 1 1 1 0 0 0 1] …
Vektor für Text 2: (dichte Darstellung, erste 10 Dimensionen) → [1 1 0 0 0 0 0 0 0 0] …
Vektor für Text 3: (dichte Darstellung, erste 10 Dimensionen) → [0 0 0 0 0 0 0 0 0 0] …
Vektor für Text 4: (dichte Darstellung, erste 10 Dimensionen) → [0 0 0 0 0 0 1 1 1 1] …

Cosine-Distanz-Matrix:
[[0.         0.73093088 1.         1.         0.15384615]
 [0.73093088 0.         1.         1.         0.86546544]
 [1.         1.         0.         1.         1.        ]
 [1.         1.         1.         0.         1.        ]
 [0.15384615 0.86546544 1.         1.         0.        ]]


In [54]:
from sklearn.decomposition import TruncatedSVD

# 4b. Schritt (optional): Dimensionsreduktion → aus (n_texte × n_ngrams) wird (n_texte × k)
svd = TruncatedSVD(n_components=50, random_state=42)
X_reduced = svd.fit_transform(X)

# Dann hat jeder Text nur noch einen 50-dimensionalen Vektor:
# X_reduced[i] ist ein dichte NumPy-Array der Länge 50 für Text i

for i in X_reduced:
    print(i)

[ 3.34782969e+00 -9.14313926e-01 -4.63112347e-17  1.17223896e-16
 -9.77786493e-01]
[ 2.18769572e+00  3.49228302e+00 -1.36050352e-15  7.66477422e-16
  1.33965510e-01]
[ 1.25048602e-15  3.28607075e-16  3.01882174e+00 -1.97147542e+00
  5.96501105e-16]
[-7.10017328e-17 -5.28421157e-16  1.97147542e+00  3.01882174e+00
  4.63181730e-16]
[ 3.16563737e+00 -1.44649710e+00 -7.98172645e-16 -1.19833758e-17
  9.41480822e-01]


In [63]:
from collections import defaultdict
import string

# Sample text data: sentences
corpus = [
    "Python is amazing and fun.",
    "Python is not just fun but also powerful.",
    "Learning Python is fun!",
]
# Function to preprocess text
def preprocess(text):
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))
    # Tokenize: split the text into words
    tokens = text.split()
    return tokens

# Apply preprocessing to the sample corpus
processed_corpus = [preprocess(sentence) for sentence in corpus]
print(processed_corpus)

[['python', 'is', 'amazing', 'and', 'fun'], ['python', 'is', 'not', 'just', 'fun', 'but', 'also', 'powerful'], ['learning', 'python', 'is', 'fun']]


In [64]:
# Initialize an empty set for the vocabulary
vocabulary = set()

# Build the vocabulary
for sentence in processed_corpus:
    vocabulary.update(sentence)

# Convert to a sorted list
vocabulary = sorted(list(vocabulary))
print("Vocabulary:", vocabulary)

Vocabulary: ['also', 'amazing', 'and', 'but', 'fun', 'is', 'just', 'learning', 'not', 'powerful', 'python']


In [66]:
def create_bow_vector(sentence, vocab):
    vector = [0] * len(vocab)  # Initialize a vector of zeros
    for word in sentence:
        if word in vocab:
            idx = vocab.index(word)  # Find the index of the word in the vocabulary
            vector[idx] += 1  # Increment the count at that index
    return vector
# Create BoW vector for each sentence in the processed corpus
bow_vectors = [create_bow_vector(sentence, vocabulary) for sentence in processed_corpus]
print("Bag of Words Vectors:")
for vector in bow_vectors:
    print(vector)


Bag of Words Vectors:
[0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1]
[1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1]
[0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1]


In [73]:
print(set(nltk_text[:50]))

{'States', 'for', 'of', '.', 'restrictions', 'Tragödie', 'world', 'away', 'use', 'anyone', 'Gutenberg', 'Faust', 'You', 'eBook', 'no', 'may', 'the', 'ebook', 'Teil', 'anywhere', 'at', 'whatsoever', ':', '\ufeffThe', 'copy', 'parts', ',', 'is', 'and', 'almost', 'it', 'zweiter', 'give', 'most', 'other', 'Project', 'or', 'cost', 'Der', 'with', 'in', 'This', 'United'}


In [25]:
nltk_text.concordance("Welt") #Auflistung des entsprechenden Suchwortes

Displaying 25 of 52 matches:
. -- In Dämmerschein liegt schon die Welt erschlossen , Der Wald ertönt von ta
esetz gesetzlich überwaltet Und eine Welt des Irrtums sich entfaltet . Der rau
h selber schützt . So will sich alle Welt zerstückeln , Vernichtigen , was sic
 wütend hausen , Schon ist die halbe Welt vertan ; Es sind noch Könige da drau
o fehlt 's nicht irgendwo auf dieser Welt ? Dem dies , dem das , hier aber feh
Zum Silber Gold , dann ist es heitre Welt ; Das übrige ist alles zu erlangen :
 Mit ihren hunderttausend Possen Die Welt ein einzig großer Tor . GÄRTNERINNEN
mal mich vergessen , Wär ' es um die Welt mir bang ; Stunden zählen , Jahre me
 in jeder Richtung Flöh ' ich zu der Welt hinaus ; Doch von drüben droht Verni
Zur Einsamkeit ! -- Da schaffe deine Welt . KNABE LENKER : So acht ' ich mich 
, Da ihm doch rein und ungestört Die Welt dort oben allein gehört . GNOMEN : D
 : Auch kommt er an ! -- Das All der Welt Wird vorgestellt Im großen Pan . Ihr
tz in deinen Händen Kom

In [26]:
nltk_text.similar("Welt") #Worte die Ähnlich verwendet werden wie der String

ich herr schlüssel welle königin zukunft not uns vor auf wenn wo nach
flut glieder schönste sich es hügel bist


In [9]:
nltk_text.common_contexts(["Welt", "Menschen"])

die_zu der_und


In [36]:
fdist1 = FreqDist(nltk_text)
print("fdist1 most common:",fdist1.most_common(100))

fdist1 most common: [(',', 5381), ('.', 2156), (':', 1306), ('!', 1040), ('und', 765), (';', 760), ("'", 657), ('ich', 648), ('der', 622), ('die', 576), ('zu', 558), ('sich', 508), ('nicht', 478), ('Und', 415), ('ist', 404), ("'s", 403), ('?', 381), ('sie', 365), ('das', 360), ('in', 355), ('es', 351), ('ein', 332), ('Die', 319), ('du', 284), ('den', 274), ('er', 271), ('--', 263), ('Der', 253), ('Das', 251), ('mich', 243), ('wie', 232), ('dem', 226), ('so', 226), ('mir', 221), ('wir', 213), ('an', 206), ('mit', 196), ('im', 195), ('MEPHISTOPHELES', 188), ('von', 187), ('auf', 187), ('uns', 183), ('ihr', 180), ('Doch', 180), ('the', 171), ('Ich', 168), ('doch', 167), ('Wie', 155), ('Was', 152), ('So', 146), ('sind', 145), ('auch', 143), ('was', 135), ('des', 135), ('nur', 130), ('FAUST', 129), ('Ein', 127), ('euch', 126), ('dich', 121), ('of', 111), ('Sie', 111), ('schon', 110), ('dir', 110), ('man', 109), ('aus', 108), ('Du', 106), ('In', 106), ('zum', 103), ('hier', 101), ('als', 101

In [13]:
fdist = FreqDist(len(w) for w in nltk_text) #FreqDist zählt die Häufigkeit wie oft welche längen vorkommen
print(fdist)

<FreqDist with 23 samples and 42174 outcomes>


In [23]:
sentences = nltk.sent_tokenize(text[1000:]) #start from the 1000th char and tokenize in sentences
len(sentences) #number of sentences
print("Die ersten 3 Sätze sind:",sentences[:3])


Die ersten 3 Sätze sind: ['sse\n  http://gutenberg.aol.de erreichbar.', 'This book was generously provided by the German Gutenberg Projekt,\n  which can be found at the web address http://gutenberg.aol.de/.', 'Faust:  Der Tragödie zweiter Teil\n\n  von Johann Wolfgang von Goethe\n\n  1.']


In [ ]:
nltk_text_clean = [t for t in nltk_text if t.isalpha()]  # Entferne Satzzeichen/Zahlen
avg_sen_len = len(nltk_text_clean) / len(sentences) #average sentence length
avg_sen_len

13.776153625680712

In [102]:
import spacy
from textstat.textstat import textstatistics
import math

# Splits the text into sentences, using 
# Spacy's sentence segmentation which can 
# be found at https://spacy.io/usage/spacy-101
def break_sentences(text):
    nlp = spacy.load('en_core_web_sm')
    doc = nlp(text)
    return list(doc.sents)

# Returns Number of Words in the text
def word_count(text):
    sentences = break_sentences(text)
    words = 0
    for sentence in sentences:
        words += len([token for token in sentence])
    return words

# Returns the number of sentences in the text
def sentence_count(text):
    sentences = break_sentences(text)
    return len(sentences)

# Returns average sentence length
def avg_sentence_length(text):
    words = word_count(text)
    sentences = sentence_count(text)
    average_sentence_length = float(words / sentences)
    return average_sentence_length

# Textstat is a python package, to calculate statistics from 
# text to determine readability, 
# complexity and grade level of a particular corpus.
# Package can be found at https://pypi.python.org/pypi/textstat
def syllables_count(word):
    return textstatistics().syllable_count(word)

# Returns the average number of syllables per
# word in the text
def avg_syllables_per_word(text):
    print("Hier kommt der Text:",text)
    syllable = syllables_count(text)
    print("Das sind die Silben:",syllable)
    words = word_count(text)
    print("Anzahl der Wörter:",words)
    ASPW = float(syllable) / float(words)
    print("Ergebnis von Silben / Wörter:",ASPW)
    return round(ASPW, 1)

# Return total Difficult Words in a text
def difficult_words(text):
    
    nlp = spacy.load('en_core_web_sm')
    doc = nlp(text)
    # Find all words in the text
    words = []
    sentences = break_sentences(text)
    for sentence in sentences:
        words += [str(token) for token in sentence]

    # difficult words are those with syllables >= 2
    # easy_word_set is provide by Textstat as 
    # a list of common words
    diff_words_set = set()
    
    for word in words:
        syllable_count = syllables_count(word)
        if word not in nlp.Defaults.stop_words and syllable_count >= 2:
            diff_words_set.add(word)

    return len(diff_words_set)

# A word is polysyllablic if it has more than 3 syllables
# this functions returns the number of all such words 
# present in the text
def poly_syllable_count(text):
    count = 0
    words = []
    sentences = break_sentences(text)
    for sentence in sentences:
        words += [token for token in sentence]
    

    for word in words:
        syllable_count = syllables_count(word)
        if syllable_count >= 3:
            count += 1
    return count


def flesch_reading_ease(text):
    """
        Implements Flesch Formula:
        Reading Ease score = 206.835 - (1.015 × ASL) - (84.6 × ASW)
        Here,
          ASL = average sentence length (number of words 
                divided by number of sentences)
          ASW = average word length in syllables (number of syllables 
                divided by number of words)
    """
    print("Avg Sentence Length:",avg_sentence_length(text))
    print("Avg Syllables per Word:",avg_syllables_per_word(text))
    FRE = 180 - avg_sentence_length(text) - float(58.5 * avg_syllables_per_word(text))
    return round(FRE, 2)


def gunning_fog(text):
    per_diff_words = (difficult_words(text) / word_count(text) * 100) + 5
    grade = 0.4 * (avg_sentence_length(text) + per_diff_words)
    return grade


def smog_index(text):
    """
        Implements SMOG Formula / Grading
        SMOG grading = 3 + ?polysyllable count.
        Here, 
           polysyllable count = number of words of more
          than two syllables in a sample of 30 sentences.
    """

    if sentence_count(text) >= 3:
        poly_syllab = poly_syllable_count(text)
        SMOG = (1.043 * (30*(poly_syllab / sentence_count(text)))**0.5) \
                + 3.1291
        return round(SMOG, 1)
    else:
        return 0


def dale_chall_readability_score(text):
    """
        Implements Dale Challe Formula:
        Raw score = 0.1579*(PDW) + 0.0496*(ASL) + 3.6365
        Here,
            PDW = Percentage of difficult words.
            ASL = Average sentence length
    """
    words = word_count(text)
    # Number of words not termed as difficult words
    count = word_count - difficult_words(text)
    if words > 0:

        # Percentage of words not on difficult word list

        per = float(count) / float(words) * 100
    
    # diff_words stores percentage of difficult words
    diff_words = 100 - per

    raw_score = (0.1579 * diff_words) + \
                (0.0496 * avg_sentence_length(text))
    
    # If Percentage of Difficult Words is greater than 5 %, then;
    # Adjusted Score = Raw Score + 3.6365,
    # otherwise Adjusted Score = Raw Score

    if diff_words > 5:       

        raw_score += 3.6365
        
    return round(score, 2)

In [67]:
engl_elementary = "Lena had a small dog. One day, it ran away. She looked in the park, the street, the shop and the store. She called its name. Then she heard a bark. Her dog was under a bench! She hugged it and smiled. Lena was so happy to find her dog."
engl_highschool = "Lena owned a playful little dog that vanished one afternoon. Worried, she searched the park, streets, and nearby shops, calling its name. Finally, she heard a familiar bark beneath a bench. There it was—muddy but safe. Lena laughed and hugged it tightly, grateful to have found her best friend."
engl_college = "Lena’s dog went missing one afternoon, prompting an anxious search through local parks, busy sidewalks, and nearby storefronts. Though her voice echoed unanswered for hours, a sudden bark beneath a bench halted her despair. Reunited, she embraced the muddy creature, overwhelmed by the fragile relief of recovering something irreplaceable."
engl_prof = "When Lena’s canine companion disappeared, a disquieting solitude enveloped her, catalyzing an exhaustive traversal of urban and liminal spaces. Her relentless invocation yielded silence—until an inconspicuous bark from beneath a bench disrupted her grief. In that moment, reunion transcended relief, illustrating the delicate interplay between loss, perseverance, and emotional resilience."

In [72]:
print("Grundschüler:",gunning_fog(engl_elementary))
print("Oberschüler:",gunning_fog(engl_highschool))
print("Studierende:",gunning_fog(engl_college))
print("Professoren:",gunning_fog(engl_prof))

Grundschüler: 7.017213114754099
Oberschüler: 15.992258064516129
Studierende: 27.493785310734463
Professoren: 29.62150537634409


In [8]:
# Text für Grundschüler (einfache Wörter, klare Sätze)
text_grundschule = (
    "Lena hatte einen kleinen Hund. Eines Tages lief er weg. "
    "Sie suchte überall: im Park, auf der Straße und beim Laden. "
    "Sie rief seinen Namen. Niemand antwortete. Sie war traurig. "
    "Dann hörte sie ein Bellen. Es kam von einer Bank. "
    "Sie ging näher. Da war ihr Hund! Er war schmutzig, aber okay. "
    "Lena war froh. Sie nahm ihn auf den Arm. "
    "Sie sagte: ‚Ich hab dich vermisst!‘ "
    "Sie ging mit ihm nach Hause. Sie gab ihm Futter. "
    "Dann spielten sie im Garten. Sie war sehr glücklich."
)

# Text für Abiturienten (mittlere Komplexität, variierter Satzbau)
text_abitur = (
    "Lena besaß einen kleinen, lebhaften Hund. Eines Nachmittags verschwand er plötzlich. "
    "Verzweifelt suchte sie in Parks, auf Straßen und bei Geschäften, rief laut seinen Namen. "
    "Niemand hatte ihn gesehen. Ihre Sorge wuchs. "
    "Plötzlich hörte sie ein vertrautes Bellen unter einer Parkbank. "
    "Vorsichtig näherte sie sich – und da war er! "
    "Verschmutzt, aber unverletzt sprang er auf sie zu. "
    "Erleichtert fiel sie auf die Knie und umarmte ihn. "
    "Sie versprach, besser aufzupassen. "
    "Zuhause angekommen, bekam er ein Bad, Futter und Streicheleinheiten. "
    "Lena war überglücklich, ihren besten Freund wiederzuhaben."
)

# Text für Studierende (gehobene Sprache, komplexere Strukturen)
text_studierende = (
    "Lena besaß einen Hund, der ihr sehr ans Herz gewachsen war. "
    "Eines Tages bemerkte sie seine Abwesenheit und begab sich auf eine ausgedehnte Suche. "
    "Sie durchstreifte Parks, Straßen und Geschäfte, rief seinen Namen immer wieder. "
    "Mit wachsender Unruhe irrte sie umher, bis ein Bellen unter einer Bank sie innehalten ließ. "
    "Dort fand sie das Tier – erschöpft, verschmutzt, aber wohlauf. "
    "Ein Gefühl tiefer Erleichterung erfasste sie. "
    "Nach der Heimkehr badete sie ihn, gab ihm Futter und umarmte ihn oft. "
    "Die Erfahrung ließ sie über Verantwortung, Verlust und Bindung nachdenken."
)

# Text für Wissenschaftler (hohe sprachliche Komplexität, abstrakte Begriffe)
text_professoren = (
    "Als Lenas vertrauter Vierbeiner auf unerklärliche Weise verschwand, initiierte sie eine systematische Suchaktion, "
    "die urbane wie periphere Areale umfasste. Ihre vokalen Aufrufe verhallten zunächst ergebnislos, "
    "bis ein auditiver Reiz – ein spezifisches Bellen – unter einer Parkbank ihre Aufmerksamkeit erregte. "
    "Die anschließende Wiedervereinigung war von intensiver emotionaler Resonanz geprägt. "
    "Das Tier, wenngleich physisch unversehrt, wies Spuren der Exposition auf. "
    "In der Nachbetrachtung reflektierte Lena über die Fragilität zwischenmenschlicher (und tierischer) Bindungen, "
    "über die anthropologische Bedeutung von Fürsorge und über Verlust als Konstituente affektiver Existenz. "
    "Die Episode gewann retrospektiv beinahe paradigmatischen Charakter."
)

In [74]:
# Ausgabe mit Gunning-Fog-Index
print("Grundschüler:", gunning_fog(text_grundschule))
print("Abiturienten:", gunning_fog(text_abitur))
print("Studierende:", gunning_fog(text_studierende))
print("Wissenschaftler:", gunning_fog(text_professoren))

Grundschüler: 9.954285714285716
Abiturienten: 15.286292834890967
Studierende: 17.535714285714285
Wissenschaftler: 25.142857142857142


In [103]:
flesch_reading_ease(text_grundschule)

Avg Sentence Length: 5.6
Hier kommt der Text: Lena hatte einen kleinen Hund. Eines Tages lief er weg. Sie suchte überall: im Park, auf der Straße und beim Laden. Sie rief seinen Namen. Niemand antwortete. Sie war traurig. Dann hörte sie ein Bellen. Es kam von einer Bank. Sie ging näher. Da war ihr Hund! Er war schmutzig, aber okay. Lena war froh. Sie nahm ihn auf den Arm. Sie sagte: ‚Ich hab dich vermisst!‘ Sie ging mit ihm nach Hause. Sie gab ihm Futter. Dann spielten sie im Garten. Sie war sehr glücklich.
Das sind die Silben: 104
Anzahl der Wörter: 112
Ergebnis von Silben / Wörter: 0.9285714285714286
Avg Syllables per Word: 0.9
Hier kommt der Text: Lena hatte einen kleinen Hund. Eines Tages lief er weg. Sie suchte überall: im Park, auf der Straße und beim Laden. Sie rief seinen Namen. Niemand antwortete. Sie war traurig. Dann hörte sie ein Bellen. Es kam von einer Bank. Sie ging näher. Da war ihr Hund! Er war schmutzig, aber okay. Lena war froh. Sie nahm ihn auf den Arm. Sie sagte: ‚

121.75

In [80]:
flesch_reading_ease(engl_elementary)

122.96

In [99]:
print("Flesch Elementary:",flesch_reading_ease(text_grundschule))
print("Flesch College:",flesch_reading_ease(text_studierende))

Avg Sentence Length: 5.6
Avg Word Length: 0.9
Flesch Elementary: 121.75
Avg Sentence Length: 13.125
Avg Word Length: 1.2
Flesch College: 96.67


In [67]:
from textstat.textstat import textstatistics

bsp = "Wenn eine Maus hunger hat, rennt sie im Wald herum und sucht sich etwas zu fressen. Dort findet sie so einiges, was sie gerne mag. Besonders lecker findet sie kleine Insekten, Samen und Nüsse." #61
bsp2 = "Die Ameise wanderte langsam über den warmen Waldboden und suchte nach Futter für ihr Volk."
bsp3 = "Am frühen Morgen, wenn der Himmel blass und grau erscheint, beginnt das leise Summen in den Zweigen, noch versteckt. Die Bäume recken ihre Äste aus und atmen Dämmerung, der Tau auf Gräsern funkelt still, als ob er Lieder weckt. Die Vögel rufen ihre Lieder durch die kühle Morgenluft, ein erster Strahl berührt das Dach, ganz sacht und ohne Hast. Ein Kind tritt aus dem alten Haus, in Stille eingehüllt, \
die Augen groß, das Herz ganz weit, vom Träumen noch erfüllt.  \
Die Welt beginnt in weichem Licht, mit Duft von feuchtem Grün, \
und jeder Schritt erzählt von Zeit, die leise neu beginnt.\
So atmet alles tief und sanft, in diesem frühen Licht,\
und flüstert dir: Vergiss die Last, das Leben trägt dich mit."
bsp_hund = "Der Hund bellt."

syllable = textstatistics().syllable_count(bsp2)
syllable

20

In [68]:
import re

def silben_zaehlen(text):
    # Alle Worte im Text extrahieren
    worte = re.findall(r'\b\w+\b', text.lower())
    vokale = "aeiouyäöü"

    def silben_in_wort(wort):
        # Zählt Gruppen von aufeinanderfolgenden Vokalen als eine Silbe
        silben = re.findall(r'[aeiouyäöü]+', wort)
        return max(1, len(silben))  # mindestens 1 Silbe

    gesamt_silben = sum(silben_in_wort(wort) for wort in worte)
    return gesamt_silben

text = "In einem alten, staubigen und dunklen Bücherzimmer saßen sieben Kinder still und lauschten einer langen, wundersamen Geschichte über ferne Länder, mutige Helden, sprechende Tiere und vergessene Zeiten."
print(silben_zaehlen(bsp2))  


26


In [69]:
import pyphen

def silben_zaehlen_mit_pyphen(text, sprache='de_DE'):
    dic = pyphen.Pyphen(lang=sprache)
    worte = text.split()
    silben_gesamt = 0

    for wort in worte:
        silben = dic.inserted(wort).split('-')
        silben_gesamt += len(silben)

    return silben_gesamt

print(silben_zaehlen_mit_pyphen(bsp2))  # z.B. 11


25


In [118]:
import spacy

nlp = spacy.load("de_core_news_sm")
doc = nlp("Der Hund läuft schnell.")

for token in doc:
    print(token.text, token.lemma_, token.pos_, token.tag_, token.dep_,
            token.shape_, token.is_alpha, token.is_stop)

Der der DET ART nk Xxx True True
Hund Hund NOUN NN sb Xxxx True False
läuft laufen VERB VVFIN ROOT xxxx True False
schnell schnell ADV ADJD mo xxxx True False
. -- PUNCT $. punct . False False


In [131]:
import spacy

nlp = spacy.load("de_core_news_sm")
doc = nlp("Sebastian läuft so schnell wie ein TomaHawk aus England von Apple aus 1998.")

for ent in doc.ents: #Named Entity Recognition 101
    print(ent.text, ent.start_char, ent.end_char, ent.label_)

Sebastian 0 9 PER
TomaHawk 35 43 MISC
England 48 55 LOC
Apple 60 65 ORG


# ------------- Modell Spielereien -------------

In [20]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string

nltk.download('punkt')
nltk.download('stopwords')

def preprocess(text):
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha()]  # Entferne Satzzeichen/Zahlen
    return tokens


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\s3phi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\s3phi\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [37]:
#Zeichen-n-Gramme (Character n-grams)
from sklearn.feature_extraction.text import CountVectorizer

texts = ["Dies ist ein kurzer Beispieltext.", 
         "Ein anderer Text vom (vielleicht) gleichen Autor."]

char_vectorizer = CountVectorizer(analyzer='char', ngram_range=(3, 5))
X_char = char_vectorizer.fit_transform(texts)
#print("x_char:",X_char)

print("Länge der Liste:",len(char_vectorizer.get_feature_names_out()))
print("Zeichen n-gramme:", char_vectorizer.get_feature_names_out()[:100])


Länge der Liste: 213
Zeichen n-gramme: [' (v' ' (vi' ' (vie' ' an' ' and' ' ande' ' au' ' aut' ' auto' ' be'
 ' bei' ' beis' ' ei' ' ein' ' ein ' ' gl' ' gle' ' glei' ' is' ' ist'
 ' ist ' ' ku' ' kur' ' kurz' ' te' ' tex' ' text' ' vo' ' vom' ' vom '
 '(vi' '(vie' '(viel' ') g' ') gl' ') gle' 'and' 'ande' 'ander' 'aut'
 'auto' 'autor' 'bei' 'beis' 'beisp' 'che' 'chen' 'chen ' 'cht' 'cht)'
 'cht) ' 'der' 'dere' 'derer' 'die' 'dies' 'dies ' 'eic' 'eich' 'eiche'
 'eicht' 'ein' 'ein ' 'ein a' 'ein k' 'eis' 'eisp' 'eispi' 'ell' 'elle'
 'ellei' 'elt' 'elte' 'eltex' 'en ' 'en a' 'en au' 'er ' 'er b' 'er be'
 'er t' 'er te' 'ere' 'erer' 'erer ' 'es ' 'es i' 'es is' 'ext' 'ext '
 'ext v' 'ext.' 'gle' 'glei' 'gleic' 'hen' 'hen ' 'hen a' 'ht)' 'ht) ']


In [45]:
#Wort-n-Gramme (Word n-grams)
word_vectorizer = CountVectorizer(analyzer='word', ngram_range=(1, 2))
X_word = word_vectorizer.fit_transform(texts)

print("Word n-grams:", word_vectorizer.get_feature_names_out()[:100])


Word n-grams: ['anderer' 'anderer text' 'autor' 'beispieltext' 'dies' 'dies ist' 'ein'
 'ein anderer' 'ein kurzer' 'gleichen' 'gleichen autor' 'ist' 'ist ein'
 'kurzer' 'kurzer beispieltext' 'text' 'text vom' 'vielleicht'
 'vielleicht gleichen' 'vom' 'vom vielleicht']


In [48]:
#Frequenz von Funktionwörtern
stop_words = set(stopwords.words('german'))

def count_function_words(tokens):
    return {word: tokens.count(word) for word in stop_words if word in tokens}

tokens = preprocess(texts[0])
print("preprocess(texts:",tokens)
fw_counts = count_function_words(tokens)
print("Funktionswörter-Frequenzen:", fw_counts)


preprocess(texts: ['dies', 'ist', 'ein', 'kurzer', 'beispieltext']
Funktionswörter-Frequenzen: {'ein': 1, 'dies': 1, 'ist': 1}


In [ ]:
#Type-Token-Ratio (TTR)
def ttr(tokens):
    return len(set(tokens)) / len(tokens) if tokens else 0

print("TTR für Text 1:", ttr(tokens))


TTR für Text 1: 1.0


Beispiel aus: https://towardsdatascience.com/basics-of-countvectorizer-e26677900f9c/

In [229]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

with open("korpus\\goethe-faust1.txt", encoding="utf8") as input:
    text = input.read()
#print("Typ:",type(text))
nltk_ding = nltk.word_tokenize(text)
#print("NLTK:",nltk_ding)
#print("TEST:",text)
#text = ["Dies ist ein kurzer Beispieltext."]
#text = [text[1750:1950]]
text = [text]
print("Typ:",type(text))
stop_words_ger = list(stopwords.words('german'))
coun_vect = CountVectorizer(stop_words=stop_words_ger)
count_matrix = coun_vect.fit_transform(text)
count_array = count_matrix.toarray()
df = pd.DataFrame(data=count_array,columns = coun_vect.get_feature_names_out())
print(df)

Typ: <class 'list'>
   000  1500  1887  20  2000  2001  2021  2229  26  30  ...  übertischten  \
0    1     1     1   1     1     1     1     1   1   1  ...             1   

   übertragen  übertreiben  übertrug  übertrümmerten  überwindet  \
0           1            1         1               1           1   

   überzulaufen  übrigbleiben  übrigbleibt  übrigen  
0             1             1            1        2  

[1 rows x 6654 columns]


# Beispiel

In [ ]:
import nltk
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

nltk.download('punkt')
nltk.download('stopwords')

# ─────────────────────────────────────
# 🔹 Schritt 1: Beispieldaten vorbereiten
# ─────────────────────────────────────
texts = [
    "Der Himmel war grau, als sie das Haus verließ. Sie trug einen Regenschirm.",
    "Das Leben ist ein langer Fluss, dachte er und blickte in die Ferne.",
    "Heute ist ein schöner Tag. Ich liebe es, spazieren zu gehen.",
    "Spaziergänge bei Regen sind etwas Besonderes - ruhig, klar, friedlich.",
]
authors = ["Autor_A", "Autor_A", "Autor_B", "Autor_B"]  # bekannte Labels

# ─────────────────────────────────────
# 🔹 Schritt 2: Feature-Extraktion definieren
# ─────────────────────────────────────

# Zeichenn-Gramme
char_vectorizer = CountVectorizer(analyzer='char', ngram_range=(3, 4))

# Wort-n-Gramme
word_vectorizer = CountVectorizer(analyzer='word', ngram_range=(1, 2))

# Kombinieren der Features
combined_vectorizer = FeatureUnion([
    ("char", char_vectorizer),
    ("word", word_vectorizer),
])

# ─────────────────────────────────────
# 🔹 Schritt 3: Modellpipeline & Training
# ─────────────────────────────────────

# Vektorisieren + Klassifikator
pipeline = Pipeline([
    ("vectorizer", combined_vectorizer),
    ("classifier", LinearSVC())
])

# Aufteilen in Training/Test
X_train, X_test, y_train, y_test = train_test_split(texts, authors, test_size=0.25, random_state=42)

# Training
pipeline.fit(X_train, y_train)

# Vorhersage & Bewertung
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

# ─────────────────────────────────────
# 🔹 Schritt 4: Funktion zur Autorenvorhersage
# ─────────────────────────────────────

def predict_author(text):
    return pipeline.predict([text])[0]

# Testaufruf:
test_text = "Sie verließ das Haus mit einem Regenschirm - es war still draußen."
print("Vorhergesagter Autor:", predict_author(test_text))


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\s3phi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\s3phi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


NameError: name 'Pipeline' is not defined